# 05. 신규 고객 재구매 특성 분석: 로지스틱 회귀

- **분석 배경**
  - 과거 신규 고객을 코호트별로 추적한 결과, 첫 구매 후 7차월에 1회 이상 구매한 고객은 그렇지 않은 고객보다 유지율이 높았습니다.
  - 그래서 7차월 구매 여부를 신규 고객이 정착했는지 판단하는 지표로 삼았습니다.
  - 이제 6차월 말 시점에 어떤 고객을 우선 관리해야 7차월 구매로 이어지는지 알고자 합니다.
- **분석 대상**
  - 2025년 7 ~ 12월에 첫 구매를 한 신규 고객 중 6차월 말(기준일)까지 탈퇴하지 않은 고객입니다.
  - 기준일 이전에 탈퇴한 고객은 7차월에 구매할 수 없으므로 분석 대상에서 제외합니다.
- **분석 질문**
  - 6차월 말까지의 구매 행동 중 어떤 특성이 7차월 구매와 관련 있는가?
- **사용 데이터**
  - `repurchase.csv`(`new_customers.csv`와 `transactions.csv`를 결합·집계한 신규 고객별 구매 행동 9,173명)
    - 만드는 과정은 `appendix_a_rfm_features.ipynb` 파일을 참조하세요.
- **분석 단위**: 신규 고객

**학습 목표**

1. 기준일을 정해 입력변수 관찰 기간과 목표변수 기간을 나누는 분석 설계를 설명할 수 있다.
2. 로지스틱 회귀 모형을 적합하고, 가능도비 검정으로 모형의 유의성을 판단할 수 있다.
3. 분산팽창지수와 상관계수로 유의하지 않은 변수의 원인을 확인하고, 변수를 제외한 최종 모형을 전체 모형과 비교할 수 있다.
4. 오즈비와 신뢰구간을 업무 언어로 해석하고, 오즈의 변화를 확률의 변화로 바꿔 계산할 수 있다.
5. 최종 모형으로 고객별 재구매 확률을 예측할 수 있다.

## 0. 준비

### 라이브러리 불러오기

In [ ]:
# 필요한 모듈을 임포트합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import hds

### 그래프 한글 폰트 설정

구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.<br>
음수 부호(−)가 네모로 깨지지 않도록 하는 설정도 함께 처리합니다.

In [ ]:
# 구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.
# 처음 실행할 때만 인터넷 연결이 필요하고, 이후에는 내려받은 파일을 다시 사용합니다.
hds.plot.set_font(font_name='Gowun Batang')

# 그래프 제목을 굵게 표시합니다.
plt.rc(group='axes', titleweight='bold')

### 데이터 불러오기

In [ ]:
# 재구매 분석용 csv 파일을 읽고 repurchase에 할당합니다.
repurchase = pd.read_csv(filepath_or_buffer='../data/repurchase.csv', encoding='utf-8-sig')

# repurchase의 처음 5행과 마지막 5행을 확인합니다.
# [참고] 데이터프레임 이름만 실행하면 처음 5행과 마지막 5행, 행 개수와 열 개수를 함께 보여줍니다.
repurchase

In [ ]:
# repurchase의 정보를 확인합니다.
repurchase.info()

## 1. 분석 설계

**분석 설계**

```
M1 ─ M2 ─ M3 ─ M4 ─ M5 ─ M6 │ M7
└──── 입력변수 관찰 ────┘  기준일  └ 목표변수: 재구매 여부
                      (M6 말일)
```

- M1은 신규 고객의 첫 구매월(2025-07 ~ 2025-12)입니다.
  - 고객마다 M1이 다르므로 기준일도 고객마다 다릅니다.
- 입력변수와 목표변수를 만드는 기준은 다음과 같습니다.
  - **입력변수**는 기준일까지의 거래로만 만들어, 데이터 누수를 방지합니다.
  - **목표변수**는 기준일 다음 달(M7)의 재구매 여부입니다.
- 기준일 **전에 탈퇴한 고객**은 M7에 재구매할 수 없으므로 분석에서 제외합니다.
  - 이미 탈퇴한 고객을 분석 대상에 포함하면 '재구매하지 않음'이 탈퇴 때문인지 행동 때문인지 구분할 수 없습니다.
- 기준일까지의 거래 데이터로 RFM(최근성·구매 횟수·구매 금액) 변수를 만드는 과정은 `부록 A`에서 자세하게 다룹니다.
  - 이번 실습에서는 그렇게 만든 `repurchase.csv`를 바로 사용합니다.

## 2. 변수 탐색과 재구매 여부별 비교

**질문**: 연속형 입력변수의 중심과 퍼짐, 최솟값과 최댓값은 어떠한가?

In [ ]:
# 연속형 입력변수 이름을 리스트로 생성합니다.
cont_cols = ['frequency', 'monetary', 'active_months', 'recency_days', 'fresh_ratio', 'discount_ratio', 'distance_km']

# 연속형 입력변수의 기술통계량을 소수점 2자리까지 반올림해서 확인합니다.
repurchase[cont_cols].describe().round(decimals=2)

**해석**
- `frequency`와 `monetary`는 평균이 중위수보다 큽니다.
  - 일부 고객이 많이 구매해서 평균을 끌어올린, 오른쪽으로 꼬리가 긴 분포입니다.
  - `frequency`의 1사분위수가 1회이므로, 고객의 4분의 1 이상은 6개월 동안 한 번만 구매했습니다.
- `monetary`는 최솟값이 0보다 크지만, `recency_days`는 최솟값이 0입니다.
  - 0에는 로그를 취할 수 없으므로, 이 차이는 3절의 로그 변환 판단에서 다시 확인합니다.
- `fresh_ratio`, `discount_ratio`는 0 ~ 1 사이의 비율입니다.
- `distance_km`는 3사분위수가 4.5km인데 최댓값은 25km로, 멀리 사는 고객이 일부 있습니다.

**질문**: 범주형 입력변수인 연령대와 주 이용 채널은 범주별로 고객이 몇 명(도수)이고, 전체에서 몇 %(상대도수)인가?

In [ ]:
# 연령대의 고유값을 확인합니다.
# [참고] unique()는 고유값을 오름차순 정렬한 결과를 반환하지 않으므로 미리 시리즈 원소를 정렬하는 것이 보기에 좋습니다.
repurchase['age_group'].sort_values().unique()

In [ ]:
# 연령대별 도수를 age_freq에 할당합니다.
age_freq = repurchase['age_group'].value_counts()

# 연령대별 상대도수를 age_prop에 할당합니다.
age_prop = repurchase['age_group'].value_counts(normalize=True)

# 연령대별 도수와 상대도수를 한 표로 정리합니다.
# [참고] sort_index()는 인덱스 기준으로 오름차순 또는 내림차순 정렬합니다.
pd.DataFrame(data={'도수': age_freq, '상대도수': age_prop}).sort_index().round(decimals=3)

In [ ]:
# 주 이용 채널의 고유값을 확인합니다.
repurchase['main_channel'].sort_values().unique()

In [ ]:
# 주 이용 채널별 도수를 channel_freq에 할당합니다.
channel_freq = repurchase['main_channel'].value_counts()

# 주 이용 채널별 상대도수를 channel_prop에 할당합니다.
channel_prop = repurchase['main_channel'].value_counts(normalize=True)

# 주 이용 채널별 도수와 상대도수를 한 표로 정리합니다.
pd.DataFrame(data={'도수': channel_freq, '상대도수': channel_prop}).sort_index().round(decimals=3)

**해석**
- 연령대는 5개 범주이고, 40대가 가장 많습니다.
  - 20대와 60대 이상은 각각 전체의 10% 남짓입니다.
- 주 이용 채널은 마트, 슈퍼, 온라인의 3개 범주이고, 마트가 절반 가까이 됩니다.
  - 슈퍼와 온라인도 각각 20% 이상이라, 고객 수가 너무 적은 범주는 없습니다.
- 4절에서 더미 변수를 만들 때 정렬 순서상 첫 범주인 20대와 마트가 기준 범주가 됩니다.
  - 기준 범주의 고객 수가 너무 적으면 계수가 불안정해지므로, 범주별 도수를 미리 확인해 둡니다.

**질문**: 0과 1로 된 입력변수와 목표변수는 1의 비율이 얼마인가?

In [ ]:
# 앱 가입 여부의 상대도수를 확인합니다.
# [참고] value_counts()는 도수 기준으로 내림차순 정렬한 결과를 반환합니다.
repurchase['app_member'].value_counts(normalize=True).round(decimals=3)

In [ ]:
# 2개 이상 채널 이용 여부의 상대도수를 확인합니다.
repurchase['multi_channel'].value_counts(normalize=True).round(decimals=3)

In [ ]:
# 목표변수인 7차월 재구매 여부의 상대도수를 확인합니다.
repurchase['repurchase_m7'].value_counts(normalize=True).round(decimals=3)

**해석**
- 0과 1로 된 변수는 1의 상대도수가 곧 평균입니다.
  - `app_member`의 평균은 앱 회원 비율, `multi_channel`의 평균은 2개 이상 채널 이용 비율로 읽습니다.
  - 앱 회원은 절반을 조금 상회하고, 2개 이상 채널을 이용하는 고객은 40% 남짓입니다.
- 7차월에 재구매한 고객의 비율은 35.0%입니다. [양성 비율]
  - 이 전체 재구매율은 7절에서 오즈비를 확률로 바꿀 때 기준 확률로 다시 사용합니다.

**질문**: 첫 구매월(코호트)별 신규 고객 수는 늘고 있는가, 줄고 있는가?

In [ ]:
# 첫 구매월별 고객 수를 월 순서로 정렬해서 확인합니다.
repurchase['cohort_ym'].value_counts().sort_index()

**해석**
- **코호트**(cohort)는 같은 시기에 같은 경험을 한 고객 집단입니다.
  - 이번 실습에서는 `cohort_ym`(첫 구매월)이 같은 고객을 하나의 코호트로 묶습니다.
  - `cohort_ym`은 입력변수로 쓰지 않습니다.
- 코호트별로 시간이 지나면서 구매율이나 유지율이 어떻게 변하는지 추적하는 분석을 **코호트 분석**이라고 합니다.
  - 이 실습의 분석 배경(7차월에 구매한 고객의 이후 유지율이 높음)도 코호트 분석으로 얻은 결과입니다.
- 2025년 7 ~ 12월 모두 한 달에 1,500명 안팎입니다.
  - 이 기간 중 신규 고객 수는 늘거나 줄지 않고 비슷하게 유지되고 있습니다.
  - 이 수는 기준일 전에 탈퇴한 고객을 뺀 것이므로 실제 첫 구매 고객 수보다 조금 적습니다.

**질문**: 재구매한 고객과 하지 않은 고객은 입력변수의 평균과 분포가 어떻게 다른가?

In [ ]:
# 0과 1로 된 변수를 포함한 수치형 입력변수 이름을 리스트로 생성합니다.
num_cols = ['frequency', 'monetary', 'active_months', 'recency_days', 'fresh_ratio', 'discount_ratio', 
            'multi_channel', 'app_member', 'distance_km']

# 재구매 여부별 입력변수 평균을 계산하고 행과 열을 바꿔서 group_mean에 할당합니다.
group_mean = repurchase.groupby(by='repurchase_m7')[num_cols].mean().T

# group_mean의 열 이름을 변경합니다.
group_mean.columns = ['재구매 X', '재구매 O']

# group_mean을 소수점 3자리까지 반올림해서 확인합니다.
group_mean.round(decimals=3)

In [ ]:
# 3행 3열의 그래프 영역을 생성합니다.
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(12, 12), dpi=100)

# 재구매 여부별 입력변수의 분포를 상자 그림으로 시각화합니다.
# [참고] 상자 안의 숫자는 재구매 여부별 평균, 빨간 점선은 전체 평균입니다.
for ax, col in zip(axes.flatten(), num_cols):
    hds.plot.box_group(data=repurchase, x='repurchase_m7', y=col, palette=['silver', 'silver'], ax=ax)
    ax.set(title=f'재구매 여부별 {col}', xlabel='M7 재구매 (0 = 안 함, 1 = 함)', ylabel=col)
plt.tight_layout()
plt.show()

**해석**
- 재구매 고객은 구매 횟수·금액·활동 월 수가 크고, `recency_days`(마지막 구매 후 경과일)이 짧습니다.
  - 이 비교는 변수 하나씩 따로 본 것입니다.
  - 구매 횟수가 많은 고객은 금액도 크므로, 어느 변수가 진짜 관련 있는지는 로지스틱 회귀 모형으로 함께 봐야 합니다.
- 상자 그림에서 `recency_days`는 두 그룹의 상자 위치가 뚜렷하게 다릅니다.
  - 최근에 구매한 고객일수록 다음 달에도 구매하는 비율이 높습니다.
  - `discount_ratio`는 재구매 고객 쪽이 조금 낮습니다.
- `frequency`, `monetary`, `distance_km`는 오른쪽으로 치우친 분포라 극단값이 많습니다.
- `multi_channel`과 `app_member`는 0과 1로 된 변수입니다.
  - 상자 모양은 같지만, 상자 안의 평균(1의 비율)에서 차이를 보입니다.

## 3. 입력변수 변환

**질문**: 오즈비를 업무 단위로 읽을 수 있도록 입력변수의 단위를 바꿉니다.

In [ ]:
# 마지막 구매 후 경과일을 30일(약 1개월) 단위로 바꿉니다.
repurchase['recency_30'] = repurchase['recency_days'] / 30

# 신선식품 비중을 10%p 단위로 바꿉니다.
repurchase['fresh_10'] = repurchase['fresh_ratio'] * 10

# 할인 결제 비중을 10%p 단위로 바꿉니다.
repurchase['discount_10'] = repurchase['discount_ratio'] * 10

# 구매 금액에 로그를 취합니다.
repurchase['log_monetary'] = np.log(repurchase['monetary'])

# 변환한 변수의 기술통계량을 확인합니다.
repurchase[['recency_30', 'fresh_10', 'discount_10', 'log_monetary']].describe().round(decimals=2)

**해석**
- 로지스틱 회귀 모형의 오즈비는 '변수가 **1단위** 증가할 때 오즈가 몇 배가 되는가'입니다.
  - 그래서 1단위의 크기가 중요합니다.
- `recency_days` 그대로면 '1일 증가할 때'의 오즈비라서 업무 판단에 쓰기에는 단위가 너무 작습니다.
  - 분석 설계가 월 단위(M1 ~ M6 관찰, M7 목표)이므로 **30일(약 1개월) 단위**로 바꿉니다.
- `fresh_ratio`, `discount_ratio`는 0 ~ 1 비율입니다.
  - '1단위 = 0%에서 100%로' 같은 비현실적 변화가 되므로 **10%p 단위**로 바꿉니다.
- `monetary`는 오른쪽으로 크게 치우쳐 있고 금액 차이는 '몇 배'로 의미가 있으므로 **로그**를 취합니다.

**질문**: `monetary`에는 로그를 취하고 `recency_days`에는 취하지 않는 이유를 구간별 로그 오즈로 확인합니다.

In [ ]:
# 구간별 고객 수, 재구매 고객 수, 재구매율, 오즈, 로그 오즈와 앞 구간 대비 로그 오즈 변화를 계산하는 함수를 정의합니다.
def logodds_by_bin(x, bins):
    table = repurchase.groupby(by=pd.cut(x=x, bins=bins, right=False), observed=True)['repurchase_m7'] \
        .agg(n='size', buy='sum')
    table['rate(p)'] = table['buy'] / table['n']
    table['odds'] = table['rate(p)'] / (1 - table['rate(p)'])
    table['log_odds'] = np.log(table['odds'])
    table['diff'] = table['log_odds'].diff()
    return table.round(decimals=3)

In [ ]:
# 마지막 구매 후 경과일을 30일 간격으로 나눠 구간별 로그 오즈를 확인합니다.
logodds_by_bin(x=repurchase['recency_days'], bins=range(0, 211, 30))

In [ ]:
# 구매 금액 구간 경계를 5만 원부터 2배씩 커지도록 생성합니다.
money_bins = [0, 50000, 100000, 200000, 400000, 800000, 1600000, 3200000]

# 구매 금액을 2배 간격으로 나눠 구간별 로그 오즈를 확인합니다.
logodds_by_bin(x=repurchase['monetary'], bins=money_bins)

**해석**
- 로그는 배수를 간격으로 바꾸는 변환입니다.
  - 로그 척도에서는 5만 → 10만 원과 40만 → 80만 원이 같은 거리입니다.
- `recency_days`는 30일마다 로그 오즈가 −0.5 ~ −0.8 정도로 비슷하게 줄어듭니다.
  - 같은 일수 간격마다 같은 크기로 변하므로, 원 척도 그대로 회귀계수 하나로 요약할 수 있습니다.
- `monetary`는 구간 폭이 5만, 10만, 20만 원으로 2배씩 커지는데도 로그 오즈는 0.6 ~ 0.8 정도로 비슷하게 늘어납니다.
  - 같은 금액이 아니라 **같은 배수(2배)**마다 같은 크기로 변하므로 로그를 취합니다.
- 원 척도로 보면 같은 1만 원 증가의 효과가 금액이 적은 구간에서 훨씬 큽니다.
  - 그래서 원 척도의 회귀계수 하나로는 이 관계를 요약할 수 없습니다.
- 마지막 구간(160만 원 이상)은 고객이 60명뿐이라 로그 오즈 변화(1.553)가 튀므로 크게 해석하지 않습니다.
- 분포가 오른쪽으로 치우쳤다는 것은 로그 변환을 검토할 신호일 뿐입니다.
  - 최종 판단은 어떤 간격마다 로그 오즈가 고르게 변하는지로 합니다.

**로그 변환 판단 기준**

| 확인 항목 | 로그를 취함 | 원 척도 유지 |
|---|---|---|
| 값의 범위 | 모두 0보다 큼 | 0이나 음수가 있음 |
| 분포 | 오른쪽으로 꼬리가 김 | 대칭이거나 범위가 정해져 있음 |
| 차이의 의미 | 몇 배 차이(금액, 매출, 면적) | 몇 단위 차이(일수, 거리, 비중) |
| 구간별 로그 오즈 | 2배 간격마다 비슷하게 변함 | 같은 간격마다 비슷하게 변함 |
| 이 실습의 예 | `monetary` | `recency_days` |

- 마지막 기준(구간별 로그 오즈)이 가장 중요합니다.
  - 앞의 세 기준은 로그 변환을 검토할 변수를 고르는 데 씁니다.
  - 선형 회귀에서는 로그 오즈 대신 구간별 목표변수 평균으로 확인합니다.
- 0이 있는 변수에 로그를 취해야 하면 `np.log1p()`처럼 값에 1을 더한 뒤 로그를 취합니다.

## 4. 로지스틱 회귀 모형 적합

**질문**: 범주형 변수를 더미 변수로 바꾸고, 입력변수 행렬과 목표변수 벡터를 만듭니다.

In [ ]:
# 로지스틱 회귀 모형에 넣을 입력변수 이름을 리스트로 생성합니다.
input_cols = ['frequency', 'log_monetary', 'active_months', 'recency_30', 'fresh_10', 'discount_10', 
              'multi_channel', 'app_member', 'distance_km', 'age_group', 'main_channel']

# 범주형 변수를 더미 변수로 변환하고 입력변수 행렬을 생성합니다.
# [참고] 기준 범주는 age_group은 20대, main_channel은 마트입니다.
X = pd.get_dummies(data=repurchase[input_cols], columns=['age_group', 'main_channel'], drop_first=True, dtype=int)

# 입력변수 행렬의 처음 5행을 확인합니다.
X.head()

In [ ]:
# 목표변수 벡터를 생성합니다.
y = repurchase['repurchase_m7'].copy()

**해석**
- 연령대는 20대, 주 이용 채널은 마트가 기준 범주입니다.
  - 각 더미 변수의 계수는 기준 범주와 비교한 차이입니다.
- [참고] 기준 범주가 아닌 범주끼리 비교할 때는 모형을 다시 적합할 필요 없이 두 계수의 차이를 보면 됩니다.
  - 예를 들어 `온라인 계수 − 슈퍼 계수`는 슈퍼 대비 온라인의 로그 오즈비이고, 이 값에 지수를 취하면 오즈비입니다.

**질문**: 모든 입력변수를 넣은 로지스틱 회귀 모형을 적합합니다.

In [ ]:
# 로지스틱 회귀 모형을 적합합니다.
logit_model = hds.stat.glm(y=y, X=X)

# 모형의 요약 정보를 확인합니다.
logit_model.summary()

**해석**
- `coef`는 로그 오즈 척도의 계수라서 바로 해석하기 어렵습니다.
  - 계수에 지수함수를 취한 **오즈비**로 바꿔서 읽으며, 7절에서 오즈비 표로 정리합니다.
- 요약 정보는 다음 순서로 읽습니다.
  - ① 모형의 유의성(가능도비 검정) → ② 회귀계수의 유의성(`P>|z|`) → ③ 의사 결정계수(`Pseudo R-squ. (CS)`)
- 로지스틱 회귀는 선형 회귀와 달리 잔차의 정규성과 등분산성을 가정하지 않습니다.
  - 목표변수가 0과 1이라 잔차가 정규분포를 따를 수 없고, 분산도 예측 확률에 따라 달라지기 때문입니다.

- 대신 다음 가정을 확인합니다.

  | 가정 | 내용 | 이번 실습의 확인 |
  |---|---|---|
  | **관측값의 독립성** | 한 고객의 결과가 다른 고객의 결과와 관계가 없음 | 고객 1명이 1행 |
  | **로그 오즈와의 선형성** | 연속형 입력변수가 로그 오즈와 직선 관계 | 3절의 구간별 로그 오즈 |
  | **심한 다중공선성 없음** | 입력변수끼리 정보가 많이 겹치지 않음 | 5절의 분산팽창지수 |
  | **충분한 표본 크기** | 목표변수의 소수 범주 수가 회귀계수 1개당 10명 이상<br>(흔히 쓰이는 **경험 규칙**) | 재구매 고객 3,206명(9,173명의 35.0%) ÷ 회귀계수 15개<br>= 1개당 약 214명으로 충분 |

  [참고] 상수항을 뺀 회귀계수 개수(15개)는 가능도비 검정의 자유도와 같습니다.

**질문**: 이 모형은 입력변수를 하나도 넣지 않은 모형(null 모형)보다 나은가? 가능도비 검정으로 모형의 유의성을 확인합니다.

In [ ]:
# null 모형과 적합한 모형의 로그 가능도를 출력합니다.
# [참고] null 모형은 입력변수가 없는 모형으로, 목표변수의 평균으로만 예측합니다.
print(f'null 모형의 로그 가능도: {logit_model.llnull:.2f}')
print(f'적합한 모형의 로그 가능도: {logit_model.llf:.2f}')

In [ ]:
# 가능도비 통계량을 계산합니다.
# [참고] 가능도비 통계량은 적합한 모형과 null 모형의 로그 가능도 차이의 2배입니다.
llr_stat = 2 * (logit_model.llf - logit_model.llnull)

# 가능도비 통계량으로 유의확률을 계산합니다.
# [참고] 자유도는 절편을 제외한 회귀계수(더미 변수 포함)의 개수입니다.
llr_pvalue = stats.chi2.sf(x=llr_stat, df=logit_model.df_model)

# 가능도비 통계량과 유의확률을 출력합니다.
print(f'가능도비 통계량 = {llr_stat:.2f}, 자유도 = {logit_model.df_model:.0f}, 유의확률 = {llr_pvalue:.4f}')

**해석**
- 유의확률이 0.05보다 작으므로 '모든 회귀계수가 0이다'라는 귀무가설을 기각합니다.
  - 입력변수 중 적어도 하나는 재구매와 관련이 있습니다.
- 어떤 변수가 관련 있는지는 이어서 회귀계수의 유의성으로 확인합니다.

**질문**: 어떤 입력변수가 재구매와 유의한 관련이 있는가? 회귀계수의 유의성을 확인합니다.

In [ ]:
# 절편을 제외한 회귀계수별 유의확률을 소수점 4자리까지 반올림해서 확인합니다.
# [참고] 요약 정보의 P>|z| 열과 같은 값입니다.
logit_model.pvalues.drop(index='const').sort_values(ascending=False).round(decimals=4)

**해석**
- 회귀계수별 유의확률은 '모회귀계수가 0이다'라는 귀무가설의 검정 결과입니다.
  - 요약 정보의 `z`는 계수를 표준오차로 나눈 값입니다.
- `log_monetary`(구매 금액), `active_months`(활동 월 수), `age_group`(연령대)의 유의확률이 0.05보다 큽니다.
  - 모회귀계수가 0이라는 귀무가설을 기각하지 못하므로, 회귀식에서 제외할지 판단해야 합니다.
- `main_channel`(주 이용 채널)은 온라인 더미만 유의확률이 0.05보다 작게 나옵니다.
  - 범주형 변수의 유의성은 6절에서 변수 단위로 확인합니다.

**질문**: 이 모형은 null 모형보다 얼마나 개선되었는가? 의사 결정계수로 확인합니다.

In [ ]:
# Cox-Snell 의사 R²를 계산합니다.
# [참고] Cox-Snell 의사 R²는 요약 정보의 Pseudo R-squ. (CS)와 같은 값입니다.
cs_r2 = 1 - np.exp(2 / logit_model.nobs * (logit_model.llnull - logit_model.llf))

# Cox-Snell 의사 R²의 최댓값을 계산합니다.
cs_max = 1 - np.exp(2 / logit_model.nobs * logit_model.llnull)

# Cox-Snell 의사 R²와 최댓값을 출력합니다.
print(f'Cox-Snell 의사 R² = {cs_r2:.4f}, 최댓값 = {cs_max:.4f}')

**해석**
- `Pseudo R-squ. (CS)`는 Cox-Snell 의사 결정계수(의사 R²)입니다.
  - 가능도비 통계량(null 모형 대비 줄어든 이탈도)을 관측값 개수로 나눠 관측값 하나당 평균으로 만들고, 지수함수로 0 ~ 1 사이 값으로 바꾼 것입니다.
  - 이 값은 입력변수를 추가한 모형이 null 모형보다 로그 가능도가 얼마나 좋아졌는지를 나타냅니다.

> $\begin{aligned}
  &\text{Cox-Snell 의사 } R^2 = 1 - \exp\left(-\frac{4272.48}{9173}\right) = 1 - 0.6277 = 0.3723 \\[10pt]
  \end{aligned}$

- 의사 R² 값이 클수록 null 모형보다 많이 개선된 것을 의미합니다.
  - 다만 선형 회귀의 R²처럼 '재구매 변동의 ○%를 설명한다'로 읽지 않습니다.
- 모형이 아무리 좋아도 1이 될 수 없습니다.
  - 완벽한 모형이라도 줄일 수 있는 이탈도는 null 모형의 이탈도($-2 \times (-5936.21) = 11872.42$)가 전부이기 때문입니다.
  - 그래서 최댓값은 null 모형의 로그 가능도로 정해지며, 위에서 출력한 최댓값과 함께 봅니다.

> $\begin{aligned}
  &\text{Cox-Snell 의사 } R^2 \text{ 최댓값} = 1 - \exp\left(-\frac{11872.42}{9173}\right) = 1 - 0.2741 = 0.7259 \\[10pt]
  \end{aligned}$

- 같은 데이터로 만든 모형을 비교할 때 사용합니다.
  - 입력변수를 넣으면 값이 줄어들지 않으므로 이 값만으로 모형을 고르지 않습니다.

## 5. 다중공선성 확인

**질문**: `log_monetary`(구매 금액), `active_months`(활동 월 수), `age_group`(연령대)는 왜 유의하지 않을까?<br>
분산팽창지수와 분산팽창지수 상위 4개 변수의 상관 히트맵으로 확인합니다.

In [ ]:
# 입력변수별 분산팽창지수를 내림차순으로 정렬해서 확인합니다.
hds.stat.vif(model=logit_model).T.sort_values(by=0, ascending=False).round(decimals=2)

In [ ]:
# 분산팽창지수 상위 4개 변수인 활동 월 수, 최근성(30일 단위), 구매 금액(로그), 구매 횟수의 상관 히트맵을 그립니다.
plt.figure(figsize=(8, 6), dpi=100)
hds.plot.corr_heatmap(data=repurchase[['active_months', 'recency_30', 'log_monetary', 'frequency']], fontsize=10)
plt.show()

**해석**
- 2절의 단순 비교에서는 재구매 고객의 `active_months`(활동 월 수), `monetary`(구매 금액)이 분명히 컸습니다.
  - 그런데 4절의 요약 정보에서는 `P>|z|`가 0.05보다 커서 유의하지 않습니다.
- 분산팽창지수 표에서는 활동 월 수, `recency_30`(최근성), 구매 금액, 구매 횟수의 분산팽창지수가 다른 변수보다 높습니다.
  - 구매를 자주 하면 금액도 크고 여러 달에 걸쳐 사게 될 뿐만 아니라 마지막 구매도 최근일 가능성이 높기 때문입니다.
- 다만 네 변수의 분산팽창지수가 모두 10보다 작아서, 선형 회귀 실습의 면적·품목 수처럼 심한 다중공선성은 아닙니다.
- 상관계수 히트맵을 보면 네 변수의 Pearson 상관계수 절댓값이 모두 높습니다.
  - 최근성은 마지막 구매 후 경과일이라 나머지 세 변수와 음의 상관이고, 나머지 세 변수끼리는 양의 상관입니다.
  - 같은 '구매 활동량'을 서로 다른 방식으로 측정한 변수라서, 한 모형에 함께 넣으면 효과를 나눠 갖습니다.
- 구매 횟수와 최근성이 같은 고객끼리 비교하면, 금액이나 활동 월 수가 추가로 알려 주는 정보가 거의 없다는 뜻입니다. [**다중공선성**]
- 최근성과 구매 횟수는 분산팽창지수가 높은 편이어도 유의합니다.
  - 분산팽창지수가 높다고 해서 항상 유의하지 않은 것은 아닙니다.
- 연령대는 가설검정 실습의 교차분석에서도 재구매율 차이가 없었습니다.
- 주 이용 채널은 슈퍼 더미는 유의하지 않고 온라인 더미는 유의합니다.
  - 가설검정 실습의 교차분석에서도 채널별 재구매율 차이가 유의했습니다.
  - 더미 변수의 유의성은 6절에서 범주형 변수 단위로 확인합니다.

## 6. 최종 모형

**질문**: `main_channel`(주 이용 채널)은 재구매와 관련 있는가? 더미 변수의 유의성을 변수 단위로 검정합니다.

요약 정보의 더미 변수별 `P>|z|`는 **해당 범주와 기준 범주의 차이**만 검정합니다.<br>
`main_channel`의 기준 범주는 마트이므로 요약 정보에는 슈퍼와 마트, 온라인과 마트의 차이만 나오고, 슈퍼와 온라인의 차이는 나오지 않습니다.<br>
기준 범주를 바꾸면 더미 변수별 유의확률도 달라집니다.<br>
우리가 알고 싶은 것은 '주 이용 채널이 재구매와 관련 있는가'이므로, 더미 변수를 묶어서 검정합니다.

1. **가설**: 귀무가설은 '슈퍼 더미와 온라인 더미의 회귀계수가 모두 0이다'입니다. 채널에 따른 차이가 없다는 것입니다.
2. **모형 비교**: 더미 변수를 모두 넣은 모형과 모두 뺀 모형을 적합합니다. 나머지 입력변수는 그대로 둡니다.
3. **가능도비 통계량**: 더미 변수를 넣어서 로그 가능도가 얼마나 좋아졌는지 계산합니다.
4. **유의확률**: 자유도가 뺀 더미 변수의 개수인 카이제곱분포에서 계산합니다.

In [ ]:
# main_channel 더미 변수 이름을 리스트로 생성합니다.
main_cols = ['main_channel_슈퍼', 'main_channel_온라인']

# main_channel 더미 변수를 모두 뺀 입력변수 행렬로 로지스틱 회귀 모형을 적합합니다.
reduced_model = hds.stat.glm(y=y, X=X.drop(columns=main_cols))

# main_channel 더미 변수를 모두 넣은 모형과 뺀 모형의 로그 가능도를 출력합니다.
print(f'채널 더미 변수를 모두 넣은 모형의 로그 가능도: {logit_model.llf:.2f}')
print(f'채널 더미 변수를 모두 뺀 모형의 로그 가능도: {reduced_model.llf:.2f}')

In [ ]:
# main_channel 더미 변수를 모두 넣은 모형과 뺀 모형의 가능도비 통계량을 계산합니다.
channel_stat = 2 * (logit_model.llf - reduced_model.llf)

# main_channel 더미 변수를 모두 넣은 모형과 뺀 모형의 가능도비 통계량으로 유의확률을 계산합니다.
# [참고] 귀무가설이 참이면 표본 크기가 클 때 가능도비 통계량은 근사적으로 카이제곱분포를 따릅니다.
# [참고] 자유도는 뺀 더미 변수의 개수입니다.
channel_pvalue = stats.chi2.sf(x=channel_stat, df=len(main_cols))

# main_channel 더미 변수를 모두 넣은 모형과 뺀 모형의 가능도비 통계량과 유의확률을 출력합니다.
print(f'가능도비 통계량 = {channel_stat:.2f}, 자유도 = {len(main_cols)}, 유의확률 = {channel_pvalue:.4f}')

**해석**
- 유의확률이 0.05보다 작으므로 귀무가설을 기각합니다.
  - 주 이용 채널은 변수 전체로 볼 때 재구매와 관련이 있습니다.
- 슈퍼 더미는 유의하지 않지만, 범주형 변수는 변수 단위로 판단하므로 더미 변수 2개를 모두 모형에 남깁니다.
  - 기준 범주를 바꿔도 이 가능도비 통계량은 달라지지 않습니다.
- `age_group`(연령대)도 같은 방법으로 검정할 수 있습니다.
- 기준 범주가 아닌 범주끼리의 차이가 유의한지는 모형을 다시 적합하지 않고 `logit_model.t_test()`로 검정할 수 있습니다.

**질문**: 유의하지 않은 입력변수를 모두 빼도 모형의 설명력이 유지되는가?

In [ ]:
# 최종 모형에서 제외할 입력변수 이름을 리스트로 생성합니다.
# [참고] 범주형 변수는 더미 변수 전체를 함께 제외합니다.
drop_cols = ['log_monetary', 'active_months', 'age_group_30대', 'age_group_40대', 'age_group_50대', 'age_group_60대 이상']

# 입력변수 행렬에서 제외할 변수를 삭제하고 X_final에 할당합니다.
X_final = X.drop(columns=drop_cols)

# 최종 로지스틱 회귀 모형을 적합합니다.
final_model = hds.stat.glm(y=y, X=X_final)

# 최종 모형의 요약 정보를 확인합니다.
final_model.summary()

In [ ]:
# 전체 모형과 최종 모형의 가능도비 통계량을 계산합니다.
drop_stat = 2 * (logit_model.llf - final_model.llf)

# 전체 모형과 최종 모형의 자유도 차이를 계산합니다.
# [참고] 자유도 차이는 제외한 입력변수(더미 변수 포함)의 개수와 같습니다.
drop_df = logit_model.df_model - final_model.df_model

# 전체 모형과 최종 모형의 가능도비 통계량으로 유의확률을 계산합니다.
# [참고] 귀무가설이 참이면 표본 크기가 클 때 가능도비 통계량은 근사적으로 카이제곱분포를 따릅니다.
drop_pvalue = stats.chi2.sf(x=drop_stat, df=drop_df)

# 전체 모형과 최종 모형의 가능도비 통계량과 유의확률을 출력합니다.
print(f'가능도비 통계량 = {drop_stat:.2f}, 자유도 = {drop_df:.0f}, 유의확률 = {drop_pvalue:.4f}')

In [ ]:
# 전체 모형과 최종 모형의 AIC와 Cox-Snell 의사 R²를 비교합니다.
aic_values = [logit_model.aic, final_model.aic]
pseudo_r2 = [logit_model.pseudo_rsquared(kind='cs'), final_model.pseudo_rsquared(kind='cs')]
pd.DataFrame(data={'AIC': aic_values, '의사 R²': pseudo_r2}, index=['전체 모형', '최종 모형']).round(decimals=4)

In [ ]:
# 최종 모형의 입력변수별 분산팽창지수를 확인합니다.
hds.stat.vif(model=final_model).T.round(decimals=2)

**해석**
- 가능도비 검정의 유의확률이 0.7911로 0.05보다 크므로 '제외한 변수의 회귀계수가 모두 0이다'라는 귀무가설을 기각하지 못합니다.
  - 이 변수들을 함께 빼도 설명력이 유의하게 줄었다고 볼 수 없습니다.
- 범주형 변수는 더미 변수별 유의확률이 아니라 변수 단위로 판단합니다.
  - 더미 변수 일부만 빼면 빠진 범주가 기준 범주에 합쳐져 나머지 계수의 의미가 바뀌기 때문입니다.
- AIC는 작을수록 좋은 모형입니다.
  - 최종 모형의 AIC가 더 작고, 의사 R²는 거의 같습니다.
- 최종 모형의 분산팽창지수는 전체 모형보다 낮습니다.
  - 최종 모형에서 제외된 변수가 재구매와 관계가 없다는 뜻은 아닙니다.
  - 구매 금액과 활동 월 수는 구매 횟수·최근성과 정보가 겹쳐서 빠진 것입니다.
- 이후 분석은 이 모형(`final_model`)을 기준으로 합니다.

## 7. 오즈비 해석 표

**질문**: 최종 모형의 회귀계수, 유의확률, 오즈비, 오즈비 95% 신뢰구간을 한 표로 정리합니다.

In [ ]:
# 최종 모형의 회귀계수 검정 결과와 오즈비 표에서 상수항을 삭제하고 or_table에 할당합니다.
or_table = hds.stat.logit_table(model=final_model).drop(index='const')

# 최종 모형의 입력변수별 회귀계수 검정 결과와 오즈비 표를 소수점 4자리까지 반올림해서 확인합니다.
or_table.round(decimals=4)

**해석**
- `odds_ratio`가 1보다 크면 해당 변수가 커질수록 재구매 가능성이 높아짐을 의미합니다.
  - 반대로 1보다 작으면 재구매 가능성이 낮아진다는 뜻이며, 1이면 관계 없음입니다.
- 오즈비 신뢰구간(`or_ci_lower` ~ `or_ci_upper`)에 **1이 들어 있으면** 유의하지 않습니다.
  - `p_value`가 0.05보다 큰 것과 같은 뜻입니다.
- 오즈(odds)는 확률이 아니라 '재구매할 확률 ÷ 재구매하지 않을 확률'입니다.
  - '오즈의 ○% 증가'는 '확률의 ○%p 증가'와 다릅니다.

**질문**: 주요 변수의 오즈비를 업무 언어 한 줄로 바꿉니다.

In [ ]:
# 변수별 1단위 증가의 의미를 딕셔너리로 생성합니다.
units = {'frequency': '구매 횟수가 1회 많으면', 
         'recency_30': '마지막 구매가 30일 더 오래되면',
         'fresh_10': '신선식품 비중이 10%p 높으면', 
         'discount_10': '할인 결제 비중이 10%p 높으면',
         'multi_channel': '2개 이상 채널을 이용하면', 
         'app_member': '앱 회원이면', 
         'distance_km': '점포까지 거리가 1km 멀면',
         'main_channel_슈퍼': '주 이용 채널이 슈퍼면(마트 대비)',
         'main_channel_온라인': '주 이용 채널이 온라인이면(마트 대비)'}

# 긴 해석 문장이 잘리지 않도록 데이터프레임 열 너비 제한을 해제합니다.
# [참고] 데이터프레임의 기본 열 너비는 50이며, 50을 초과하는 문장은 '...'으로 생략되어 표시됩니다.
pd.set_option('display.max_colwidth', None)

# 전체 재구매율을 기준 확률로 정합니다.
base_prob = y.mean()

# 기준 확률로 기준 오즈를 계산합니다.
# [참고] 오즈비가 같아도 확률 변화는 기준 확률에 따라 달라지므로 출발점을 먼저 정해야 합니다.
base_odds = base_prob / (1 - base_prob)

# 기준 확률과 기준 오즈를 소수점 4자리까지 반올림해서 출력합니다.
print(f'기준 확률 = {base_prob:.4f}, 기준 오즈 = {base_odds:.4f}')

In [ ]:
# 오즈비, 유의확률과 기준 확률로 해석 문장과 확률 변화를 계산해 rows에 추가합니다.
rows = []
for var, condition in units.items():
    odds_ratio, p_value = or_table.loc[var, ['odds_ratio', 'p_value']]
    direction = '증가' if odds_ratio > 1 else '감소'
    effect = f'오즈 약 {abs(odds_ratio - 1) * 100:.1f}% {direction}' if p_value < 0.05 else '오즈의 차이가 유의하지 않음'
    new_odds = base_odds * odds_ratio
    new_prob = new_odds / (1 + new_odds)
    rows.append({'변수': var, '오즈비': round(number=odds_ratio, ndigits=3),
                 '해석': f'{condition} {effect}',
                 '재구매 확률': f'{base_prob * 100:.1f}% → {new_prob * 100:.1f}%',
                 '확률 변화(%p)': round(number=(new_prob - base_prob) * 100, ndigits=1)})

# rows를 데이터프레임으로 변환해서 확인합니다.
pd.DataFrame(data=rows)

**해석**
- 모든 해석에는 '**다른 변수가 같다면**'이라는 조건이 붙습니다.
- 오즈비는 해당 변수가 1단위 증가할 때 기준 오즈에 곱하는 배수입니다.
  - 오즈비가 1보다 크면 오즈가 증가하고, 1보다 작으면 감소합니다.
- `해석` 열의 '오즈의 차이가 유의하지 않음'은 유의확률이 0.05 이상이라는 뜻입니다.
  - 해당 입력변수와 재구매 사이에 관계가 있다고 보기 어렵습니다.
- 오즈를 알면 `재구매 확률`을 계산할 수 있습니다.

> $\begin{aligned}
  &\text{기준 오즈} = \frac{\text{기준 확률}}{1 - \text{기준 확률}} = \frac{0.34950}{1 - 0.34950} = 0.5373 \\[10pt]
  \end{aligned}$

- `재구매 확률`과 `확률 변화(%p)`는 전체 재구매율을 기준 확률로 놓고, 오즈비만큼 오즈가 바뀌었을 때의 확률과 그 차이입니다.
  - `확률 변화(%p)`는 반올림 전 값으로 계산하므로, 표에 보이는 두 확률의 차이와 0.1 정도 다를 수 있습니다.
  - `frequency`를 예로 들면 다음 순서로 계산합니다. (기준 확률은 전체 재구매율인 0.34950으로 설정했습니다.)

> $\begin{aligned}
  &\text{새 오즈} = \text{기준 오즈} \times \text{오즈비} = 0.5373 \times 1.1152 = 0.5992 \\[10pt]
  &\text{새 확률} = \frac{\text{새 오즈}}{1 + \text{새 오즈}} = \frac{0.5992}{1 + 0.5992} = 0.375
  \end{aligned}$

- 구매 횟수가 **2회** 많으면 오즈비를 두 번 곱하므로, 오즈는 오즈비의 제곱($1.1152^2 = 1.2437$)배가 됩니다.

> $\begin{aligned}
  &\text{새 오즈} = \text{기준 오즈} \times \text{오즈비}^2 = 0.5373 \times 1.1152^2 = 0.6682 \\[10pt]
  &\text{새 확률} = \frac{\text{새 오즈}}{1 + \text{새 오즈}} = \frac{0.6682}{1 + 0.6682} = 0.401
  \end{aligned}$

- 오즈는 1회마다 같은 배수(1.1152배)로 변하지만, 확률은 0.350 → 0.375 → 0.401로 증가 폭이 2.5%p → 2.6%p로 조금 커집니다.
  - 확률이 0.5에 가까워지기 때문입니다.
- 오즈비가 같아도 기준 확률이 다른 고객은 확률 변화가 다릅니다.
  - 대체로 기준 확률이 0.5에 가까울수록 변화가 크고, 0이나 1에 가까울수록 작습니다.
- 로지스틱 회귀 모형은 입력변수가 1단위 늘 때 로그 오즈가 **항상 같은 크기로** 변한다고 가정합니다.
  - 할인 결제 비중처럼 구간에 따라 효과가 달라질 수 있는 입력변수는 오즈비를 평균적인 요약으로 읽습니다.
  - 할인 결제 비중의 곡선 형태는 재구매 예측 실습(머신러닝)의 부분 의존도 플롯으로 확인합니다.

**[참고] 로그를 취한 입력변수의 오즈비**
- `log_monetary`(구매 금액의 로그)는 최종 모형에서 제외했지만, 남았다면 오즈비를 다르게 읽어야 합니다.
  - 로그 값이 1 증가하는 것은 구매 금액이 e배(약 2.7배)가 되는 것이라 업무 언어로 쓰기 어렵습니다.
- 구매 금액이 k배가 되면 오즈는 $k^{\beta}$배가 됩니다. ($\beta$는 `log_monetary`의 회귀계수)
  - 3절에서 구매 금액을 2배 간격으로 나눴던 것처럼 '2배'를 기준으로 읽으면 편리합니다.
  - 전체 모형(`logit_model`)의 회귀계수 0.1064로 계산하면 다음과 같습니다.

> $\begin{aligned}
  &\text{구매 금액이 2배일 때의 오즈비} = 2^{0.1064} = e^{0.1064 \times \ln 2} = 1.077 \\[10pt]
  \end{aligned}$

- '다른 변수가 같다면 구매 금액이 2배인 고객은 재구매 오즈가 약 7.7% 높다'로 읽습니다.
  - 구매 횟수가 같은데 금액만 2배라면 한 번에 많이 사는 고객이라는 뜻입니다.
- 처음부터 `np.log2()`로 변환하면 오즈비 표의 값이 바로 '금액이 2배일 때의 오즈비'가 됩니다.
  - 로그의 밑을 바꿔도 유의확률은 같습니다.

## 8. 예측 확률

**질문**: 특성이 서로 다른 가상 고객(3명)의 재구매 확률은 얼마로 예측되는가?

In [ ]:
# 재구매 가능성이 높음·중간·낮음인 가상 고객 3명을 생성합니다.
profiles = pd.DataFrame(data={'frequency': [10, 4, 1], 
                              'recency_30': [15 / 30, 90 / 30, 180 / 30],
                              'fresh_10': [5.5, 4.5, 3.0], 
                              'discount_10': [1.0, 2.0, 4.0],
                              'multi_channel': [1, 0, 0], 
                              'app_member': [1, 1, 0], 
                              'distance_km': [1.5, 3.0, 8.0]},
                        index=['고객 1 (높음)', '고객 2 (중간)', '고객 3 (낮음)'])

# profiles의 열 순서를 최종 모형의 입력변수 행렬(X_final)과 같게 맞추고, 없는 열은 0으로 채웁니다.
# [참고] 주 이용 채널 더미 변수가 모두 0이므로 세 고객의 주 이용 채널은 기준 범주인 마트입니다.
profiles = profiles.reindex(columns=X_final.columns, fill_value=0)

# profiles에 상수항을 추가합니다.
profiles.insert(loc=0, column='const', value=1)

# 가상 고객별 재구매 예측 확률을 계산합니다.
final_model.predict(exog=profiles).round(decimals=3)

**해석**
- 고객 1은 자주(10회)·최근에(15일 전) 구매, 할인 결제 비중이 낮고(10%), 여러 채널을 쓰는 앱 회원입니다.
  - 고객 3은 반대 특성입니다(세 고객 모두 주 이용 채널은 마트로 가정).
  - 자주·최근에 구매하고, 할인 결제 비중이 낮은 고객 1의 예측 확률이 가장 높고, 반대 특성의 고객 3이 가장 낮습니다.
- 예측 확률은 **'재구매한다/안 한다'로 구분하기 전의 값**입니다.
  - 몇 % 이상을 '재구매 예상 고객'으로 볼지는 별도로 정해야 합니다.
  - 임계값 설정 방법과 성능 평가는 재구매 예측 실습(머신러닝)에서 다룹니다.

## 정리

- 입력변수는 기준일(M6 말일) 이전 거래만으로 만들고, 기준일 전에 탈퇴한 고객은 분석에서 제외합니다.
- 변수 단위를 30일, 10%p 등으로 바꾸면 오즈비를 업무 언어로 해석하기 쉽습니다.
- 최근성, 구매 횟수, 신선식품 비중, 할인 결제 비중, 채널 다양성, 주 이용 채널, 앱 가입, 거리가 재구매와 유의한 관련이 있었습니다.
- 구매 금액·활동 월 수는 구매 횟수·최근성과 상관이 높아, 함께 넣으면 추가 정보가 거의 없어 유의하지 않았습니다.
- 유의하지 않은 변수를 함께 빼도 설명력이 유의하게 줄지 않아, 이 변수들을 제외한 최종 모형으로 오즈비를 해석했습니다.

## 현업 전달 포인트

통계 용어 없이 설명하면 다음과 같습니다.<br>
아래 숫자는 다른 조건이 같은 평균 고객(재구매율 35.0%)을 기준으로 계산한 것입니다.

- 마지막 구매 후 30일이 더 지나면 재구매 가능성이 35.0%에서 23.0%로 떨어집니다.
  - 구매 공백이 길어지기 전에 접촉하면 재구매가 늘어나는지 시험해 볼 만합니다.
- 할인 결제 비중이 10%p 높은 고객은 재구매 가능성이 35.0%에서 19.5%로 낮습니다.
  - 할인 때문에 재구매가 줄었다고 단정할 수는 없습니다.
  - 원래 재구매 성향이 낮은 고객이 할인을 많이 썼을 수도 있기 때문입니다.
  - 할인 위주 고객에게 다른 관리 방식을 시험해 볼 것을 제안합니다.
- 신선식품 비중이 10%p 높은 고객은 재구매 가능성이 35.0%에서 41.8%로 높습니다.
  - 신선식품 강화가 재구매를 늘리는지는 A/B 테스트로 확인해야 합니다.
- 여러 채널을 이용하는 고객은 46.1%, 앱 회원은 41.4%로 재구매 가능성이 높습니다.
  - 앱 가입과 여러 채널 이용을 유도하면 재구매가 늘어나는지는 A/B 테스트로 확인해야 합니다.

## End of Document